# ============================================================
# Cell 1: Imports & Setup
# ============================================================

In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
import warnings, random, time, os
from tqdm.notebook import tqdm as tqdm_notebook
import plotly.graph_objects as go
from plotly.subplots import make_subplots
warnings.filterwarnings('ignore')

try:
    from numba import cuda, jit, prange
    GPU_AVAILABLE = cuda.is_available()
except:
    GPU_AVAILABLE = False
    from numba import jit, prange

print(f"{'='*70}")
print(f"SiO₂ Monte Carlo | {'⚡ GPU' if GPU_AVAILABLE else '💻 CPU (Numba JIT)'}")
print(f"{'='*70}")

SiO₂ Monte Carlo | 💻 CPU (Numba JIT)


# ============================================================
# Cell 2: Physical Constants & BKS Potential
# ============================================================

In [2]:

K_B = 8.617333262145e-5
COULOMB_CONST = 14.39964547842567
ATM_TO_EV_A3 = 6.242e-7
MIN_DIST = 0.8
HARD_REPULSION = 1000.0

BUCK_A = np.array([[2029.2204, 13702.905], [13702.905, 0.0]], dtype=np.float64)
BUCK_RHO = np.array([[0.343645, 0.193817], [0.193817, 1.0]], dtype=np.float64)
BUCK_C = np.array([[192.58, 54.681], [54.681, 0.0]], dtype=np.float64)
CHARGES = np.array([-1.2, 2.4], dtype=np.float64)

CUTOFF = 10.0
SKIN = 2.0
NEIGH_CUT = CUTOFF + SKIN

n_atoms_total = 10200
n_sio2 = n_atoms_total / 3
target_density = 2.2
target_volume = (n_sio2 * 60.084) / (6.022e23 * target_density) * 1e24
target_box = target_volume ** (1/3)

BOX_INIT = target_box
BOX_MIN = target_box * 0.80
BOX_MAX = target_box * 1.50

print(f"✓ BKS Potential loaded | Target box: {target_box:.2f} Å")

✓ BKS Potential loaded | Target box: 53.62 Å


# ============================================================
# Cell 3: Simulation Parameters
# ============================================================

In [3]:

N_STEPS_TOTAL = 100000

ENSEMBLE_SCHEDULE = [
    (0,      20000,  'NPT', 5000.0,  1.0),
    (20000,  40000,  'NPT', 4000.0,  1.0),
    (40000,  60000,  'NVT', 3000.0,  None),
    (60000,  75000,  'NVT', 1500.0,  None),
    (75000,  90000,  'NVE', 800.0,   None),
    (90000, 100000,  'NVE', 300.0,   None),
]

XYZ_OUTPUT_FREQ = 10000
TRAJECTORY_FREQ = 1000
HISTORY_FREQ = 100

MAX_DISP = 0.12
TARGET_ACC = 0.40
ACC_TOL = 0.08
NEIGH_UPDATE = 200
DISP_ADJUST = 1000

print(f"✓ {N_STEPS_TOTAL:,} steps | NPT→NVT→NVE")

✓ 100,000 steps | NPT→NVT→NVE


# ============================================================
# Cell 4: Energy Functions
# ============================================================

In [4]:

if GPU_AVAILABLE:
    @cuda.jit
    def energy_kernel(coords, types, neighbors, neigh_counts,
                      energies, box_length, cutoff, min_dist,
                      hard_rep, coulomb_const, charges,
                      buck_a, buck_rho, buck_c):
        i = cuda.grid(1); n = coords.shape[0]
        if i >= n: return
        xi, yi, zi = coords[i,0], coords[i,1], coords[i,2]
        ti = types[i]; qi = charges[ti]
        half = box_length/2.0; e = 0.0
        for idx in range(neigh_counts[i]):
            j = neighbors[i,idx]
            if j < 0: continue
            dx=xi-coords[j,0]; dy=yi-coords[j,1]; dz=zi-coords[j,2]
            if dx>half: dx-=box_length
            elif dx<-half: dx+=box_length
            if dy>half: dy-=box_length
            elif dy<-half: dy+=box_length
            if dz>half: dz-=box_length
            elif dz<-half: dz+=box_length
            r2=dx*dx+dy*dy+dz*dz; r=r2**0.5
            if r<min_dist: e+=hard_rep*(min_dist-r); continue
            if r>=cutoff: continue
            tj=types[j]; A=buck_a[ti,tj]; rho=buck_rho[ti,tj]; C=buck_c[ti,tj]
            eb=A*(-r/rho).__exp__()-C/(r2*r2*r2) if A>0 else 0.0
            damp=1.0-(-0.5*r).__exp__()
            ec=coulomb_const*qi*charges[tj]*damp/r
            e+=eb+ec
        energies[i]=e

def total_energy(coords, types, neigh_arr, neigh_counts, box):
    if GPU_AVAILABLE:
        n=len(coords)
        d_c=cuda.to_device(np.ascontiguousarray(coords, dtype=np.float64))
        d_t=cuda.to_device(np.ascontiguousarray(types, dtype=np.int32))
        d_n=cuda.to_device(np.ascontiguousarray(neigh_arr, dtype=np.int32))
        d_cnt=cuda.to_device(np.ascontiguousarray(neigh_counts, dtype=np.int32))
        d_e=cuda.device_array(n, dtype=np.float64)
        blk,grd=256,(n+255)//256
        energy_kernel[grd,blk](d_c,d_t,d_n,d_cnt,d_e,
            np.float64(box),np.float64(CUTOFF),np.float64(MIN_DIST),
            np.float64(HARD_REPULSION),np.float64(COULOMB_CONST),
            cuda.to_device(CHARGES),cuda.to_device(BUCK_A),
            cuda.to_device(BUCK_RHO),cuda.to_device(BUCK_C))
        cuda.synchronize()
        return np.sum(d_e.copy_to_host())/2.0
    else:
        return energy_cpu(coords, types, neigh_arr, neigh_counts, box)

@jit(nopython=True, fastmath=True, parallel=True)
def energy_cpu(coords, types, neigh, cnt, box):
    n=len(coords); e=0.0; half=box/2.0
    for i in prange(n):
        xi,yi,zi=coords[i]; ti=types[i]; qi=CHARGES[ti]
        for idx in range(cnt[i]):
            j=neigh[i,idx]
            if j<=i: continue
            dx=xi-coords[j,0]; dy=yi-coords[j,1]; dz=zi-coords[j,2]
            if dx>half: dx-=box
            elif dx<-half: dx+=box
            if dy>half: dy-=box
            elif dy<-half: dy+=box
            if dz>half: dz-=box
            elif dz<-half: dz+=box
            r=(dx*dx+dy*dy+dz*dz)**0.5
            if r<MIN_DIST: e+=HARD_REPULSION*(MIN_DIST-r); continue
            if r>=CUTOFF: continue
            tj=types[j]
            A,rho,C=BUCK_A[ti,tj],BUCK_RHO[ti,tj],BUCK_C[ti,tj]
            eb=A*np.exp(-r/rho)-C/(r**6) if A>0 else 0.0
            damp=1.0-np.exp(-0.5*r)
            e+=eb+COULOMB_CONST*qi*CHARGES[tj]*damp/r
    return e

print("✓ Energy kernels compiled")

✓ Energy kernels compiled


# ============================================================
# Cell 5: Helpers
# ============================================================

In [5]:

def build_neighbors(coords, box, cutoff):
    tree = cKDTree(coords, boxsize=box)
    pairs = tree.query_pairs(r=cutoff, output_type='ndarray')
    n = len(coords); neigh = [[] for _ in range(n)]
    for i,j in pairs: neigh[i].append(j); neigh[j].append(i)
    max_n = max(len(nl) for nl in neigh) if n>0 else 0
    arr = np.full((n, max_n), -1, dtype=np.int32)
    cnt = np.zeros(n, dtype=np.int32)
    for i in range(n):
        cnt[i] = len(neigh[i])
        if cnt[i]>0: arr[i,:cnt[i]] = neigh[i]
    return arr, cnt

def get_ensemble_params(step):
    for start, end, ens, T, P in ENSEMBLE_SCHEDULE:
        if start <= step < end:
            return ens, T, P, (step-start)/(end-start)
    return 'NVE', 300.0, None, 0.0

def save_xyz(filename, coords, types, box, energy, step, ensemble, T, cn, density):
    with open(filename, 'w') as f:
        f.write(f"{len(coords)}\n")
        f.write(f"Step={step} Ensemble={ensemble} T={T:.0f}K E={energy:.1f}eV "
                f"Box={box:.4f}A Density={density:.3f}g/cm3 CN={cn:.3f}\n")
        for i in range(len(coords)):
            sym = "O" if types[i]==0 else "Si"
            f.write(f"{sym} {coords[i,0]:.6f} {coords[i,1]:.6f} {coords[i,2]:.6f}\n")

def compute_cn(coords, types, box, r_cut=2.2):
    si = np.where(types==1)[0]
    if len(si)==0: return 0.0
    tree = cKDTree(coords, boxsize=box)
    return np.mean([sum(1 for n in tree.query_ball_point(coords[i],r_cut) if types[n]==0) for i in si])

def get_density(box):
    return n_sio2*60.084/(6.022e23*box**3)*1e24

print("✓ Helpers ready")

✓ Helpers ready


# ============================================================
# Cell 6: MC Moves
# ============================================================

In [6]:

def move_atom(coords, types, box, e_cur, T, neigh, cnt, max_d):
    i = random.randrange(len(coords))
    old = coords[i].copy()
    coords[i] = (coords[i] + np.random.uniform(-max_d, max_d, 3)) % box
    e_new = total_energy(coords, types, neigh, cnt, box)
    de = e_new - e_cur
    if de <= 0 or random.random() < np.exp(-de/(K_B*T)):
        return True, e_new
    coords[i] = old
    return False, e_cur

def move_volume(coords, box, types, e_cur, T, P, neigh, cnt):
    n = len(coords)
    sc = np.exp(0.0003*(2*random.random()-1))
    nb = box*sc
    if nb < BOX_MIN or nb > BOX_MAX:
        return coords, box, e_cur, False, neigh, cnt
    nc = coords*sc
    nn, ncnt = build_neighbors(nc, nb, NEIGH_CUT)
    ne = total_energy(nc, types, nn, ncnt, nb)
    dE = ne-e_cur; dV = nb**3-box**3
    dH = dE + P*ATM_TO_EV_A3*dV - n*K_B*T*np.log(sc**3)
    if dH <= 0 or random.random() < np.exp(-dH/(K_B*T)):
        return nc, nb, ne, True, nn, ncnt
    return coords, box, e_cur, False, neigh, cnt

print("✓ MC moves ready")

✓ MC moves ready


# ============================================================
# Cell 7: MCSimulation Class
# ============================================================

In [7]:

class MCSimulation:
    def __init__(self, coords, types, box_init):
        self.coords = coords % box_init
        self.types = types
        self.box = box_init
        self.n_atoms = len(coords)
        self.output_dir = "mc_output"
        os.makedirs(self.output_dir, exist_ok=True)
        self.neigh, self.cnt = build_neighbors(self.coords, self.box, NEIGH_CUT)
        self.e_current = total_energy(self.coords, self.types, self.neigh, self.cnt, self.box)
        self.history = {'step':[],'energy':[],'temperature':[],'box':[],'density':[],'cn':[],'ensemble':[]}
        self.trajectory_frames = []
        self.best_e = self.e_current
        self.best_coords = self.coords.copy()
        self.best_box = self.box
        self.acc_moves = 0; self.att_moves = 0
        self.acc_vol = 0; self.att_vol = 0
        self.max_disp = MAX_DISP
        self.total_time = 0
    
    def run(self):
        start_time = time.time()
        cn = compute_cn(self.coords, self.types, self.box)
        rho = get_density(self.box)
        print(f"🚀 Initial: E={self.e_current:.0f}eV | Box={self.box:.2f}Å | ρ={rho:.3f} | CN={cn:.3f}\n")
        
        pbar = tqdm_notebook(total=N_STEPS_TOTAL, desc="MC 100k", unit="step")
        for step in range(1, N_STEPS_TOTAL+1):
            ens, T, P, prog = get_ensemble_params(step)
            
            if ens == 'NPT' and step % 100 == 0:
                self.att_vol += 1
                self.coords, self.box, self.e_current, ok, self.neigh, self.cnt = \
                    move_volume(self.coords, self.box, self.types, self.e_current, T, P, self.neigh, self.cnt)
                if ok: self.acc_vol += 1
            else:
                self.att_moves += 1
                ok, self.e_current = move_atom(self.coords, self.types, self.box,
                                               self.e_current, T, self.neigh, self.cnt, self.max_disp)
                if ok: self.acc_moves += 1
            
            if step % HISTORY_FREQ == 0:
                self._log_step(step, ens, T)
            if step % DISP_ADJUST == 0:
                self._adjust_displacement()
            if step % NEIGH_UPDATE == 0:
                self._rebuild_neighbors()
            if step % XYZ_OUTPUT_FREQ == 0:
                self._save_checkpoint(step, ens, T)
            if step % TRAJECTORY_FREQ == 0:
                self.trajectory_frames.append({
                    'coords':self.coords.copy(),'box':self.box,
                    'step':step,'ensemble':ens,'temperature':T})
            
            if step % 100 == 0:
                ar = self.acc_moves/max(self.att_moves,1)
                elapsed = time.time()-start_time
                sps = step/elapsed; eta = (N_STEPS_TOTAL-step)/sps
                eta_str = f"{eta/3600:.1f}h" if eta>3600 else f"{eta/60:.0f}min" if eta>60 else f"{eta:.0f}s"
                for s,e,en,_,_ in ENSEMBLE_SCHEDULE:
                    if s <= step < e: phase_str = f"{en}({(step-s)/(e-s)*100:.0f}%)"; break
                else: phase_str = "NVE"
                pbar.set_postfix_str(f"{phase_str}|T={T:.0f}K|E={self.e_current:.0f}|box={self.box:.2f}|acc={ar:.0%}|ETA={eta_str}")
            pbar.update(1)
        pbar.close()
        
        self.total_time = time.time()-start_time
        print(f"\n✅ Done in {self.total_time/60:.1f}min")
        self._save_final()
    
    def _log_step(self, step, ens, T):
        cn = compute_cn(self.coords, self.types, self.box)
        rho = get_density(self.box)
        for k,v in zip(['step','energy','temperature','box','density','cn','ensemble'],
                       [step,self.e_current,T,self.box,rho,cn,ens]):
            self.history[k].append(v)
        if self.e_current < self.best_e:
            self.best_e = self.e_current
            self.best_coords = self.coords.copy()
            self.best_box = self.box
    
    def _adjust_displacement(self):
        if self.att_moves == 0: return
        ar = self.acc_moves/self.att_moves
        if ar < TARGET_ACC-ACC_TOL: self.max_disp *= 0.95
        elif ar > TARGET_ACC+ACC_TOL: self.max_disp *= 1.05
        self.max_disp = max(0.02, min(self.max_disp, 0.8))
        self.acc_moves = 0; self.att_moves = 0
    
    def _rebuild_neighbors(self):
        self.neigh, self.cnt = build_neighbors(self.coords, self.box, NEIGH_CUT)
        self.e_current = total_energy(self.coords, self.types, self.neigh, self.cnt, self.box)
    
    def _save_checkpoint(self, step, ens, T):
        cn = compute_cn(self.coords, self.types, self.box)
        rho = get_density(self.box)
        fn = f"{self.output_dir}/step_{step:06d}_{ens}_T{T:.0f}K.xyz"
        save_xyz(fn, self.coords, self.types, self.box, self.e_current, step, ens, T, cn, rho)
    
    def _save_final(self):
        cn_f = compute_cn(self.best_coords, self.types, self.best_box)
        rho_f = get_density(self.best_box)
        save_xyz(f"{self.output_dir}/BEST_STRUCTURE.xyz", self.best_coords, self.types,
                 self.best_box, self.best_e, N_STEPS_TOTAL, 'FINAL', 300, cn_f, rho_f)
        pd.DataFrame(self.history).to_csv(f"{self.output_dir}/simulation_history.csv", index=False)
        
        with open(f"{self.output_dir}/full_trajectory.xyz", 'w') as f:
            for frame in self.trajectory_frames:
                f.write(f"{self.n_atoms}\n")
                cn = compute_cn(frame['coords'], self.types, frame['box'])
                rho = get_density(frame['box'])
                f.write(f"Step={frame['step']} Ens={frame['ensemble']} T={frame['temperature']:.0f}K "
                       f"Box={frame['box']:.4f}A Density={rho:.3f}g/cm3 CN={cn:.3f}\n")
                for i in range(self.n_atoms):
                    sym = "O" if self.types[i]==0 else "Si"
                    f.write(f"{sym} {frame['coords'][i,0]:.6f} {frame['coords'][i,1]:.6f} {frame['coords'][i,2]:.6f}\n")
        
        print(f"\n📁 Files in '{self.output_dir}/': BEST_STRUCTURE.xyz, simulation_history.csv, full_trajectory.xyz")
        self._generate_plots(cn_f, rho_f)
    
    def _generate_plots(self, cn_f, rho_f):
        print("📊 Generating plots...")
        df = pd.DataFrame(self.history); s = df['step'].values
        fig, axes = plt.subplots(3,2,figsize=(18,16))
        
        ax=axes[0,0]; ax.plot(s,df['energy'],'b-',lw=0.5,alpha=0.5)
        if len(df)>50:
            w=max(10,len(df)//100)
            ax.plot(s[w-1:],np.convolve(df['energy'],np.ones(w)/w,mode='valid'),'r-',lw=2)
        for start,_,_,_,_ in ENSEMBLE_SCHEDULE:
            if start>0: ax.axvline(start,color='gray',ls=':',alpha=0.3)
        ax.set(xlabel='Step',ylabel='Energy (eV)',title='Energy'); ax.grid(alpha=0.2)
        
        ax=axes[0,1]; ax.plot(s,df['temperature'],'r-',lw=1.5)
        ax.fill_between(s,0,df['temperature'],alpha=0.1,color='red')
        ax.set(xlabel='Step',ylabel='T (K)',title='Temperature'); ax.grid(alpha=0.2)
        
        ax=axes[1,0]; ax.plot(s,df['box'],'g-',lw=0.8)
        ax.axhline(target_box,color='gray',ls='--',alpha=0.5)
        ax.set(xlabel='Step',ylabel='Box (Å)',title='Box Size'); ax.grid(alpha=0.2)
        
        ax=axes[1,1]; ax.plot(s,df['density'],'purple',lw=1)
        ax.axhline(target_density,color='gray',ls='--',alpha=0.5)
        ax.set(xlabel='Step',ylabel='Density (g/cm³)',title='Density'); ax.grid(alpha=0.2)
        
        ax=axes[2,0]; ax.plot(s,df['cn'],'orange',lw=1.5)
        ax.axhline(4.0,color='gray',ls='--',alpha=0.5)
        ax.set(xlabel='Step',ylabel='CN',title='Coordination',ylim=(3.5,4.5)); ax.grid(alpha=0.2)
        
        from scipy.spatial import cKDTree as cKD
        def rdf(c,b,dr=0.1,rmax=10):
            n=len(c);rho=n/b**3;bins=int(rmax/dr);hist=np.zeros(bins)
            tree=cKD(c,boxsize=b)
            for ii,jj in tree.query_pairs(r=rmax,output_type='ndarray'):
                rv=(c[ii]-c[jj]+b/2)%b-b/2;r=np.sqrt(np.sum(rv**2))
                if r<rmax:
                    idx=int(r/dr)
                    if idx<bins: hist[idx]+=2
            rv2=(np.arange(bins)+0.5)*dr;sv=4*np.pi*rv2**2*dr
            with np.errstate(divide='ignore',invalid='ignore'):
                g=np.where(sv>0,hist/(n*(n/b**3)*sv),0.0)
            return rv2,g
        
        ax=axes[2,1]; r,gr=rdf(self.best_coords,self.best_box)
        ax.plot(r,gr,'b-',lw=1.5)
        for pos,name,clr in [(1.62,'Si-O','red'),(2.65,'O-O','blue')]:
            ax.axvline(pos,color=clr,ls='--',alpha=0.5,label=f'{name}')
        ax.set(xlabel='r (Å)',ylabel='g(r)',title='RDF',xlim=(0,8)); ax.legend(fontsize=8); ax.grid(alpha=0.2)
        
        plt.suptitle(f'SiO₂ MC {N_STEPS_TOTAL:,} Steps | E={self.best_e:.0f}eV | ρ={rho_f:.3f} | CN={cn_f:.2f}',
                     fontsize=14,fontweight='bold')
        plt.tight_layout(); plt.savefig(f"{self.output_dir}/analysis_plots.png",dpi=200); plt.show()

print("✅ MCSimulation class ready")

✅ MCSimulation class ready


# ============================================================
# Cell 8: Run Simulation
# ============================================================

In [ ]:
print("📂 Loading Q-alpha.pdb...")

filename = "SiO2_21A_3d.pdb"
coords_list = []
types_list = []
box_a = box_b = box_c = None

with open(filename, 'r') as f:
    for line in f:
        if line.startswith('ATOM') or line.startswith('HETATM'):
            atom_name = line[12:16].strip()
            x = float(line[30:38]); y = float(line[38:46]); z = float(line[46:54])
            
            if atom_name[0] == 'S' or atom_name[:2] == 'SI':
                types_list.append(1)
            elif atom_name[0] == 'O':
                types_list.append(0)
            else:
                continue
            coords_list.append([x, y, z])
        
        if line.startswith('CRYST1'):
            box_a = float(line[6:15]); box_b = float(line[15:24]); box_c = float(line[24:33])

coords = np.array(coords_list)
types = np.array(types_list, dtype=np.int32)

# Wrap into box
if box_a is not None:
    box_x, box_y, box_z = box_a, box_b, box_c
else:
    span = coords.max(axis=0) - coords.min(axis=0)
    box_x, box_y, box_z = span[0]*1.02, span[1]*1.02, span[2]*1.02

coords[:,0] %= box_x; coords[:,1] %= box_y; coords[:,2] %= box_z

n_atoms = len(coords)
n_si = np.sum(types==1)
n_o = np.sum(types==0)
n_sio2 = n_si

# Use average box for cubic MC
box_cubic = (box_x + box_y + box_z) / 3

# Scale to cubic box
scale = box_cubic / np.array([box_x, box_y, box_z])
coords = coords * scale % box_cubic

print(f"📂 Loaded {n_atoms:,} atoms ({n_si} Si + {n_o} O)")
print(f"   PDB box: {box_x:.2f}×{box_y:.2f}×{box_z:.2f} → cubic: {box_cubic:.2f} Å")
print(f"   Density: {n_sio2*60.084/(6.022e23*box_cubic**3)*1e24:.3f} g/cm³\n")

sim = MCSimulation(coords, types, box_cubic)
sim.run()

cn_final = compute_cn(sim.best_coords, sim.types, sim.best_box)
rho_final = get_density(sim.best_box)
print(f"\n{'='*60}")
print(f"🏁 DONE | Time: {sim.total_time/60:.1f}min | E: {sim.best_e:,.0f}eV | ρ: {rho_final:.3f} | CN: {cn_final:.3f}")
print(f"{'='*60}")

📂 Loading Q-alpha.pdb...
📂 Loaded 5,184 atoms (1728 Si + 3456 O)
   PDB box: 42.79×42.79×42.79 → cubic: 42.79 Å
   Density: 2.201 g/cm³

🚀 Initial: E=-29573eV | Box=42.79Å | ρ=2.201 | CN=4.009



MC 100k:   0%|          | 0/100000 [00:00<?, ?step/s]